In [1]:
# importando tudo necessário para trabalhar com spans e componentes
import re
import spacy
from spacy.tokens import Span
from spacy.language import Language
from spacy.util import filter_spans

In [2]:
# pega o txt
with open("historico_wow.txt", "r") as f:
    text = f.read()

In [3]:
pattern = r"(Shambling Horrors|Raging Spirits|Vile Spirits|Frozen Orbs|Val'kyr Shadowguard)"

for match in re.finditer(pattern, text):
    print(match) # exibindo os spans encontrados pelo regex

<re.Match object; span=(906, 925), match="Val'kyr Shadowguard">
<re.Match object; span=(944, 958), match='Raging Spirits'>
<re.Match object; span=(979, 991), match='Vile Spirits'>
<re.Match object; span=(1146, 1163), match='Shambling Horrors'>
<re.Match object; span=(2352, 2366), match='Raging Spirits'>
<re.Match object; span=(2511, 2522), match='Frozen Orbs'>
<re.Match object; span=(5961, 5972), match='Frozen Orbs'>
<re.Match object; span=(7467, 7481), match='Raging Spirits'>
<re.Match object; span=(7533, 7545), match='Vile Spirits'>
<re.Match object; span=(7617, 7631), match='Raging Spirits'>
<re.Match object; span=(8700, 8714), match='Raging Spirits'>


In [4]:
# convertendo esses matches em entidades do spaCy usando char_span
nlp = spacy.blank("en")
doc = nlp(text)

original_ents = list(doc.ents)
mwt_ents = []

for match in re.finditer(pattern, doc.text):
    start, end = match.span()
    span = doc.char_span(start, end) # convertendo indices de caracteres em span
    print(span)
    if span is not None:
        mwt_ents.append((span.start, span.end, span.text))

for ent in mwt_ents:
    start, end, name = ent
    span = Span(doc, start, end, label="ADD") # rotulando como ADD termo de raid
    original_ents.append(span)

doc.ents = original_ents

for ent in doc.ents:
    print(ent.text, ent.label_)

Val'kyr Shadowguard
Raging Spirits
Vile Spirits
Shambling Horrors
Raging Spirits
Frozen Orbs
Frozen Orbs
Raging Spirits
Vile Spirits
Raging Spirits
Raging Spirits
Val'kyr Shadowguard ADD
Raging Spirits ADD
Vile Spirits ADD
Shambling Horrors ADD
Raging Spirits ADD
Frozen Orbs ADD
Frozen Orbs ADD
Raging Spirits ADD
Vile Spirits ADD
Raging Spirits ADD
Raging Spirits ADD


In [5]:
print(mwt_ents) # conferindo a lista de spans

[(219, 221, "Val'kyr Shadowguard"), (227, 229, 'Raging Spirits'), (235, 237, 'Vile Spirits'), (270, 272, 'Shambling Horrors'), (505, 507, 'Raging Spirits'), (535, 537, 'Frozen Orbs'), (1195, 1197, 'Frozen Orbs'), (1481, 1483, 'Raging Spirits'), (1493, 1495, 'Vile Spirits'), (1510, 1512, 'Raging Spirits'), (1713, 1715, 'Raging Spirits')]


In [10]:
from spacy.util import filter_spans

# empacotando a logica da aula num componente customizado para reutilizar
@Language.component("add_ner")
def add_ner(doc):
    pattern = r"(Shambling Horrors|Raging Spirits|Vile Spirits|Frozen Orbs|Val'kyr Shadowguard)"
    original_ents = list(doc.ents)
    mwt_ents = []
    for match in re.finditer(pattern, doc.text):
        start, end = match.span()
        span = doc.char_span(start, end)
        if span is not None:
            mwt_ents.append((span.start, span.end, span.text))
    for ent in mwt_ents:
        start, end, name = ent
        span = Span(doc, start, end, label="ADD")
        original_ents.append(span)
    filtered = filter_spans(original_ents) # Adicionado para resolver sobreposições
    doc.ents = filtered
    return doc

In [7]:
nlp2 = spacy.blank("en")
nlp2.add_pipe("add_ner") # adicionando o componente ao pipeline vazio
doc2 = nlp2(text)
print(doc2.ents)

(Val'kyr Shadowguard, Raging Spirits, Vile Spirits, Shambling Horrors, Raging Spirits, Frozen Orbs, Frozen Orbs, Raging Spirits, Vile Spirits, Raging Spirits, Raging Spirits)


In [8]:
# o modelo ja reconheceu alguns desses termos de forma errado ent por isso priorizando o sapn mais longo
@Language.component("ability_ner")
def ability_ner(doc):
    pattern = r"(Bloodlust|Aura Mastery|Hammer of Justice|Holy Wrath|Anti-Magic Shell|Divine Shield|Pain Suppression|Divine Hymn|Righteous Fury|Ardent Defender)"
    original_ents = list(doc.ents)
    mwt_ents = []
    for match in re.finditer(pattern, doc.text):
        start, end = match.span()
        span = doc.char_span(start, end)
        if span is not None:
            mwt_ents.append((span.start, span.end, span.text))
    for ent in mwt_ents:
        start, end, name = ent
        span = Span(doc, start, end, label="ABILITY")
        original_ents.append(span)
    filtered = filter_spans(original_ents) # resolvendo sobreposições antes de salvar
    doc.ents = filtered
    return doc

In [11]:
nlp3 = spacy.load("en_core_web_sm") # carregando modelo completo
nlp3.add_pipe("add_ner")
nlp3.add_pipe("ability_ner") # dois componentes customizados no mesmo pipeline

doc3 = nlp3(text)

for ent in doc3.ents:
    print(ent.text, ent.label_) # adds e habilidades agora aparecem com rótulos

the World First Kill of Heroic ORG
25 CARDINAL
Lich King PERSON
Paragon ORG
3 CARDINAL
1 CARDINAL
2 CARDINAL
1 Balance CARDINAL
2 CARDINAL
5 CARDINAL
Paladin - 1 Protection ORG
2 Holy PERSON
2 Retribution
3 QUANTITY
1 CARDINAL
1 CARDINAL
3 CARDINAL
2 CARDINAL
1 CARDINAL
2 Affliction CARDINAL
25 CARDINAL
Lich King Strategy PERSON
Paragon PERSON
first ORDINAL
Lich King PERSON
26.3.2010 DATE
That week DATE
5% PERCENT
3.3.3 CARDINAL
the previous Wednesday DATE
Frost DK PERSON
Boomkin GPE
5% PERCENT
Lich King PERSON
103.9 CARDINAL
10% PERCENT
Val'kyr Shadowguard ORG
3M Health Points
Raging Spirits ORG
4.1 CARDINAL
Vile Spirits ADD
3 CARDINAL
2 CARDINAL
1 CARDINAL
2 CARDINAL
Shambling Horrors ADD
Horrors NORP
Ghouls PERSON
Protection Paladin PERSON
Lazeil GPE
the Lich King FAC
Ghouls PERSON
Lazeil’s PERSON
Horrors NORP
Lazeil PERSON
Ghouls PERSON
Sejta GPE
Protection Paladin PERSON
Horrors NORP
Ardent Defender PERSON
Lazeil PERSON
two CARDINAL
Enrage ORG
Retribution Paladin PERSON
Pain Suppr